# 02 — Grain, Candidate Keys & Duplicates

**Objetivo:** probar hipótesis de granularidad y claves sin asumir que una columna es PK solo porque se llame ID.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Preguntas
- ¿Qué representa una fila en cada fuente?
- ¿Qué claves candidatas son realmente únicas?
- ¿Cuántos nulls tienen?
- ¿Cuál es la multiplicidad máxima?

**Interpretación:** una clave no es aceptada por nombre; es aceptada por evidencia + semántica de negocio.

In [ ]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

In [ ]:
from argos.profiling.keys import (
    analyze_all_candidate_keys,
    analyze_all_normalization_collisions,
)
from argos.common.reports import write_report

cfg = PROJECT_CONFIG["profiling"]

# 1) Audit lossy normalization collisions explicitly. A collision is a
#    schema finding, not a reason to abort key analysis unless the candidate
#    key itself is ambiguous.
normalization_collisions = analyze_all_normalization_collisions(
    FILES,
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    normalization_collisions,
    paths.reports / "profiling" / "column_normalization_collisions.parquet",
)

# 2) Analyze configured candidate keys. Unrelated normalization collisions
#    no longer block the analysis.
key_analysis = analyze_all_candidate_keys(
    FILES,
    DATASETS,
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    key_analysis,
    paths.reports / "profiling" / "key_analysis.parquet",
)

display(normalization_collisions)
key_analysis


In [ ]:
# Candidatos que podrían ser PK técnica cuando son no-null.
key_analysis.loc[
    key_analysis.get("is_unique_when_nonnull", False).fillna(False),
    ["dataset", "candidate_key", "row_count", "null_key_rate", "uniqueness_ratio", "max_multiplicity"]
]

## Trabajo manual requerido
Completa `docs/data-dictionary/*.md` con una frase explícita del grain, por ejemplo:
`1 fila = 1 evento de modificación contractual`.

Si el análisis contradice la documentación fuente, conserva ambas evidencias y no corrijas silenciosamente.

## M1 remediation — duplicate structure

Uniqueness ratios alone do not tell us whether a repeated key is an exact duplicate or a genuinely versioned/multi-row entity. This diagnostic fingerprints the complete Bronze row for the two grains that block Silver design.


In [ ]:
from argos.profiling.grain import analyze_duplicate_structure, top_duplicate_keys

grain_probes = [
    ("processes", ("id_del_proceso",)),
    ("additions", ("id_contrato", "fecharegistro", "tipo")),
]

grain_resolution = pd.DataFrame([
    analyze_duplicate_structure(
        dataset,
        FILES[dataset],
        key_columns,
        sample_size=cfg["csv_sample_size"],
        temp_directory=paths.root / ".cache" / "duckdb",
    )
    for dataset, key_columns in grain_probes
])

write_report(
    grain_resolution,
    paths.reports / "profiling" / "grain_resolution.parquet",
)

grain_resolution


In [ ]:
duplicate_examples = pd.concat(
    [
        top_duplicate_keys(
            dataset,
            FILES[dataset],
            key_columns,
            sample_size=cfg["csv_sample_size"],
            limit=25,
            temp_directory=paths.root / ".cache" / "duckdb",
        )
        for dataset, key_columns in grain_probes
    ],
    ignore_index=True,
)

write_report(
    duplicate_examples,
    paths.reports / "profiling" / "duplicate_key_examples.parquet",
)

duplicate_examples
